# GDP loader

Loads World Bank GDP (current US$), indicator `NY.GDP.MKTP.CD`, into the MySQL
`gdp` table.

**Period:** 2020 through the current year. As of now this means 2020-2026.

The notebook:
- requests the indicator for all countries/economies
- keeps only economies that exist in `dim_country`, which excludes World Bank
  aggregates such as World, regions and income groups
- stores `country_iso3` alongside the country label - the label is what the spec
  asks for, the ISO3 code is what the model actually joins on
- skips unpublished `NULL` observations and counts them
- uses an upsert so the notebook can be rerun safely
- prints a year-level validation summary after loading

Shared connection, retry and transform helpers live in `etl.py`.

Run `project_1.sql` and `dimensions.ipynb` first: the foreign key
`fk_gdp_country` needs `dim_country` populated.

MySQL credentials are read from the existing `.env` file.

In [1]:
%pip install -q requests python-dotenv mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from datetime import date
from decimal import Decimal

import etl


# ============================================================
# CONFIGURATION
# ============================================================

START_YEAR = 2020
END_YEAR = date.today().year   # 2026 now; updates automatically in future years

INDICATOR = "NY.GDP.MKTP.CD"   # GDP (current US$)


def to_money(value):
    """
    Convert via str to Decimal, never through float.

    gdp_in_usd is DECIMAL(20,2). Going through a binary float first would
    introduce a rounding error before MySQL ever sees the number.
    """
    return Decimal(str(value))


# ============================================================
# UPSERT
#
# The unique key is (year, country_iso3), so a rerun refreshes the value and
# the country label instead of inserting a duplicate.
#
# The aliased-row form (AS new / new.col) rather than VALUES(): MySQL
# deprecated VALUES() inside ON DUPLICATE KEY UPDATE in 8.0.20. Both forms are
# collapsed into a single multi-row statement by mysql-connector's
# executemany - measured on a 5,000-row load, not assumed.
# ============================================================

INSERT_QUERY = """
INSERT INTO gdp
    (`year`, country, country_iso3, gdp_in_usd)
VALUES
    (%s, %s, %s, %s) AS new
ON DUPLICATE KEY UPDATE
    country    = new.country,
    gdp_in_usd = new.gdp_in_usd
"""

In [3]:
# ============================================================
# EXTRACT -> TRANSFORM -> LOAD
# ============================================================

session = etl.make_session()
connection = etl.connect_mysql()

try:
    valid_iso3 = etl.load_country_keys(connection)
    print("Countries available in dim_country:", len(valid_iso3))

    if not valid_iso3:
        raise RuntimeError(
            "dim_country is empty - run project_1.sql and dimensions.ipynb first."
        )

    # ----------------------------
    # EXTRACT
    # ----------------------------
    api_rows = etl.fetch_worldbank_indicator(
        session, INDICATOR, START_YEAR, END_YEAR
    )
    print("GDP observations returned by API:", len(api_rows))

    # ----------------------------
    # TRANSFORM
    # ----------------------------
    gdp_rows, skipped_null, skipped_unknown = etl.transform_indicator(
        api_rows, valid_iso3, to_money
    )

    print("GDP rows ready for MySQL:", len(gdp_rows))
    print("Skipped - value not published yet (NULL):", skipped_null)
    print("Skipped - aggregate or unknown economy:", skipped_unknown)

    # ----------------------------
    # LOAD
    # ----------------------------
    loaded = etl.upsert(connection, INSERT_QUERY, gdp_rows)
    print("\nGDP load completed successfully. Rows processed:", loaded)

except Exception:
    print("\nGDP load failed.")
    raise

finally:
    connection.close()
    print("MySQL connection closed.")

Countries available in dim_country: 217
GDP observations returned by API: 1590
GDP rows ready for MySQL: 1219
Skipped - value not published yet (NULL): 83
Skipped - aggregate or unknown economy: 288

GDP load completed successfully. Rows processed: 1219
MySQL connection closed.


In [4]:
# ============================================================
# VALIDATION
#
# Coverage is ragged on purpose - the World Bank simply has not published
# every country for every year. Printing the count per year makes that
# visible instead of letting it show up later as a blank chart.
#
# Only GDP is checked here. The cross-table check ("country-years with
# population but no GDP") lives in the vw_data_quality view: running it from
# this notebook would depend on whether the population loader had already
# run, and would quietly report nonsense if it had not.
# ============================================================

connection = etl.connect_mysql()

try:
    print("GDP rows currently stored by year:")
    etl.print_rows(connection, """
        SELECT
            `year`,
            COUNT(*)          AS countries,
            MIN(gdp_in_usd)   AS min_gdp,
            MAX(gdp_in_usd)   AS max_gdp
        FROM gdp
        GROUP BY `year`
        ORDER BY `year`
    """)

    # Which years are short, measured against the best-covered year.
    print()
    print("GDP coverage gap per year, vs the best-covered year:")
    etl.print_rows(connection, """
        SELECT
            `year`,
            COUNT(*) AS countries,
            (SELECT MAX(c)
               FROM (SELECT COUNT(*) AS c FROM gdp GROUP BY `year`) AS per_year
            ) - COUNT(*) AS missing_vs_best_year
        FROM gdp
        GROUP BY `year`
        ORDER BY `year`
    """)

finally:
    connection.close()

GDP rows currently stored by year:
   (2020, 210, Decimal('52302514.99'), Decimal('21375281000000.00'))
   (2021, 210, Decimal('61597447.16'), Decimal('23725645000000.00'))
   (2022, 209, Decimal('54104146.32'), Decimal('26054614000000.00'))
   (2023, 204, Decimal('50491930.92'), Decimal('27811517000000.00'))
   (2024, 200, Decimal('56752265.80'), Decimal('29298013000000.00'))
   (2025, 186, Decimal('57345434.72'), Decimal('30769700000000.00'))

GDP coverage gap per year, vs the best-covered year:
   (2020, 210, 0)
   (2021, 210, 0)
   (2022, 209, 1)
   (2023, 204, 6)
   (2024, 200, 10)
   (2025, 186, 24)
